## Indlæser masterdatasættet

In [4]:
import pandas as pd
import numpy as np
import os

BASE = "/Users/PC/Documents/Speciale/Analyse/Speciale"
CLEAN_DIR = os.path.join(BASE, "clean")

master = pd.read_csv(os.path.join(CLEAN_DIR, "master_monthly.csv"))
master["month"] = pd.to_datetime(master["month"])

### Regner afkast på verdensindeks (MSCI) og globalt statsobligationsindeks

In [5]:
# ============================================================
# Indeksafkast — omregn niveauer til månedsafkast
# ============================================================
# NDDUWI = globalt aktieindeks (MSCI World net TR), LGTRTRUU = globalt statsobl.
master["eq_world_ret"]  = master["NDDUWI"].pct_change()
master["bond_world_ret"] = master["LGTRTRUU"].pct_change()

# tjek
print(master[["month","NDDUWI","eq_world_ret","LGTRTRUU","bond_world_ret"]].head(3))
print(master[["eq_world_ret","bond_world_ret"]].describe())

       month    NDDUWI  eq_world_ret  LGTRTRUU  bond_world_ret
0 2010-01-31  2680.472           NaN  190.9851             NaN
1 2010-02-28  2718.258      0.014097  191.4753        0.002567
2 2010-03-31  2886.601      0.061930  188.5217       -0.015425
       eq_world_ret  bond_world_ret
count    199.000000      199.000000
mean       0.009912        0.000566
std        0.041819        0.018515
min       -0.132343       -0.058405
25%       -0.016042       -0.009530
50%        0.014379        0.001195
75%        0.032709        0.012677
max        0.127862        0.049404


In [6]:
# ============================================================
# Valutaafkast — harmonisér quote-retning
# Mål: "afkast på at holde fremmed valuta mod USD" for alle par
# ============================================================
# EURUSD, GBPUSD: allerede "USD per fremmed valuta" → pct_change = fremmed valutas afkast mod USD
master["eur_ret"] = master["EURUSD"].pct_change()
master["gbp_ret"] = master["GBPUSD"].pct_change()

# USDJPY, USDCHF, USDDKK: "fremmed valuta per USD" → invertér, så vi får fremmed valutas afkast mod USD
master["jpy_ret"] = (1 / master["USDJPY"]).pct_change()
master["chf_ret"] = (1 / master["USDCHF"]).pct_change()
master["dkk_ret"] = (1 / master["USDDKK"]).pct_change()

# tjek: alle skal nu betyde "fremmed valuta styrkes mod USD = positivt afkast"
fx = ["eur_ret","gbp_ret","jpy_ret","chf_ret","dkk_ret"]
print(master[fx].describe().T[["mean","std","min","max"]])
print("\nKorrelationer mellem valutaafkast:")
print(master[fx].corr().round(2))

             mean       std       min       max
eur_ret -0.000597  0.024124 -0.074319  0.075237
gbp_ret -0.000553  0.023563 -0.080922  0.055157
jpy_ret -0.002508  0.026736 -0.084222  0.077062
chf_ret  0.001705  0.026119 -0.112530  0.080526
dkk_ret -0.000618  0.024062 -0.073974  0.074151

Korrelationer mellem valutaafkast:
         eur_ret  gbp_ret  jpy_ret  chf_ret  dkk_ret
eur_ret     1.00     0.70     0.33     0.71     1.00
gbp_ret     0.70     1.00     0.23     0.54     0.70
jpy_ret     0.33     0.23     1.00     0.45     0.33
chf_ret     0.71     0.54     0.45     1.00     0.71
dkk_ret     1.00     0.70     0.33     0.71     1.00
